# Imports

In [1]:
import numpy as np
from src.get_gender_data import get_founder_gender
from src.scrape_wiki import scrape_wiki_table
from src.get_gender_data import get_founder_gender, _parse_names
from src.filter_df import filter_df

import pandas as pd
import os

# Functions

In [2]:
def rescue_unknowns(df, founder_col, gender_col):
    """Re-process only rows where gender is unknown or contains unknown."""
    mask = df[gender_col].apply(lambda x: 'unknown' in str(x).lower() if x is not None else True)
    unknown_count = mask.sum()

    print(f"Attempting to rescue {unknown_count} unknowns in {gender_col}...")

    # Targeted update
    df.loc[mask, gender_col] = df.loc[mask, founder_col].apply(lambda x: get_founder_gender(x, use_web_search=True))

    new_unknown_count = df[gender_col].apply(lambda x: 'unknown' in str(x).lower()).sum()
    print(f"Resolution complete. Unknowns remaining: {new_unknown_count} (Rescued {unknown_count - new_unknown_count})")
    return df

In [3]:


# Ensure output directory exists
os.makedirs("data", exist_ok=True)

def all_unknown_genders(founder_string):
    """Skip gender prediction and return 'unknown' for every founder."""
    if founder_string is None or (isinstance(founder_string, float) and pd.isna(founder_string)):
        return []
    return ['unknown'] * len(_parse_names(str(founder_string).strip()))

In [4]:
url = "https://en.wikipedia.org/wiki/List_of_unicorn_startup_companies"
idx = 2  # Changed to target the main list of unicorn companies

currentUnicorns_df = scrape_wiki_table(url, idx, "data/current_unicorns.csv")
currentUnicorns_df.to_csv("data/current_unicorns.csv")

pastUnicorns_df = scrape_wiki_table(url, idx+1, "data/past_unicorns.csv")
pastUnicorns_df.to_csv("data/past_unicorns.csv")

Found 4 wikitable(s)
Saved 618 rows to data/current_unicorns.csv
Found 4 wikitable(s)
Saved 208 rows to data/past_unicorns.csv


In [5]:
currentUnicorns_df = pd.read_csv("data/current_unicorns.csv", index_col=0)
pastUnicorns_df = pd.read_csv("data/past_unicorns.csv", index_col=0)



In [6]:
print("Predicting genders for current unicorns...")

currentUnicorns_df['Founder_Genders'] = filter_df(currentUnicorns_df, "Founder(s)", all_unknown_genders)

# Save intermediate result
currentUnicorns_df.to_csv("data/current_unicorns_with_gender.csv", index=False)

df_companies_founders_gender = currentUnicorns_df[['Company', 'Founder_Genders', 'Industry']].copy()
# Note: get_founder_gender now returns normalized 'male', 'female', 'unknown'
print("Done.")

Predicting genders for current unicorns...
Done.


In [7]:
# display(currentUnicorns_df)
# display(pastUnicorns_df)

will run for several minutes:

In [8]:
currentUnicorns_df['Founder_Genders'] = filter_df(currentUnicorns_df, "Founder(s)", all_unknown_genders)


In [ ]:
print("Processing Current Unicorns...")
currentUnicorns_df['Founder_Genders'] = filter_df(currentUnicorns_df, "Founder(s)", all_unknown_genders)
currentUnicorns_df = rescue_unknowns(currentUnicorns_df, 'Founder(s)', 'Founder_Genders')

print("\nProcessing Past Unicorns...")
pastUnicorns_df['Founder_Genders'] = filter_df(pastUnicorns_df, "Founders", all_unknown_genders)
pastUnicorns_df = rescue_unknowns(pastUnicorns_df, 'Founders', 'Founder_Genders')

# Save the improved data
currentUnicorns_df.to_csv('data/current_unicorns.csv', index=False)
pastUnicorns_df.to_csv('data/past_unicorns.csv', index=False)

Processing Current Unicorns...
Attempting to rescue 207 unknowns in Founder_Genders...


# Diagramme

In [ ]:
mask = (currentUnicorns_df['Founder_Genders'].apply(lambda x: len(x) > 0))
currentUnicorns_with_gender_df = currentUnicorns_df.loc[mask]

mask = (pastUnicorns_df['Founder_Genders'].apply(lambda x: len(x) > 0))
pastUnicorns_with_gender_df = pastUnicorns_df.loc[mask]

mask = (currentUnicorns_df['Founder_Genders'].apply(lambda x: len(x) == 0))
currentUnicorns_without_gender_df = currentUnicorns_df.loc[mask]

mask = (pastUnicorns_df['Founder_Genders'].apply(lambda x: len(x) == 0))
pastUnicorns_without_gender_df = pastUnicorns_df.loc[mask]


In [ ]:
# display(currentUnicorns_with_gender_df)
# display(currentUnicorns_without_gender_df)
#
# display(pastUnicorns_with_gender_df)
# display(pastUnicorns_without_gender_df)


In [ ]:
currentUnicorns_with_gender_df['Valuation(US$ billions)'] = pd.to_numeric(
    currentUnicorns_with_gender_df['Valuation(US$ billions)'],
    errors='coerce'   # wrong values -> NaN
)
current_unicorn_w_gender_valuation = currentUnicorns_with_gender_df['Valuation(US$ billions)'].astype(float).sum()

currentUnicorns_without_gender_df['Valuation(US$ billions)'] = pd.to_numeric(
    currentUnicorns_without_gender_df['Valuation(US$ billions)'],
    errors='coerce'   # wrong values -> NaN
)
current_unicorn_wo_gender_valuation = currentUnicorns_without_gender_df['Valuation(US$ billions)'].astype(float).sum()





pastUnicorns_with_gender_df['Last valuation(US$billions)'] = pd.to_numeric(
    pastUnicorns_with_gender_df['Last valuation(US$billions)'],
    errors='coerce'   # wrong values -> NaN
)
past_unicorn_w_gender_valuation = pastUnicorns_with_gender_df['Last valuation(US$billions)'].astype(float).sum()

pastUnicorns_without_gender_df['Last valuation(US$billions)'] = pd.to_numeric(
    pastUnicorns_without_gender_df['Last valuation(US$billions)'],
    errors='coerce'   # wrong values -> NaN
)
past_unicorn_wo_gender_valuation = pastUnicorns_without_gender_df['Last valuation(US$billions)'].astype(float).sum()

print(current_unicorn_w_gender_valuation, current_unicorn_wo_gender_valuation, past_unicorn_w_gender_valuation, past_unicorn_wo_gender_valuation)

In [ ]:
type(currentUnicorns_with_gender_df['Founder_Genders'])

sorted_series1 = currentUnicorns_with_gender_df['Founder_Genders'].sort_values(key=lambda x: x.apply(len))
sorted_series2 = pastUnicorns_with_gender_df['Founder_Genders'].sort_values(key=lambda x: x.apply(len))

sorted_series_all = [sorted_series1, sorted_series2]

In [ ]:
female_stats = []
male_stats = []


for a_sorted_number,sorted_series in enumerate(sorted_series_all):
    total_female_founders = []
    total_male_founders = []
    founder_number = []

    founder_gender_len = 0

    temp_m = 0
    temp_f = 0

    for a, founder_genders in enumerate(sorted_series):
        # print(a,founder_genders, len(founder_genders))
        if len(founder_genders) != founder_gender_len:
            founder_number.append(founder_gender_len)
            total_female_founders.append(temp_f)
            total_male_founders.append(temp_m)

            temp_m = 0
            temp_f = 0

            temp_m += founder_genders.count("male")
            temp_f += founder_genders.count("female")

            founder_gender_len = len(founder_genders)
            # print(temp_m, temp_f, " 1")



        else:
            temp_m += founder_genders.count("male")
            temp_f += founder_genders.count("female")

            # print(temp_m, temp_f, " 2")

    total_female_founders.append(temp_f)
    total_male_founders.append(temp_m)
    founder_number.append(founder_gender_len)

    total_male_founders = total_male_founders[1:]
    male_stats.append(total_male_founders[1:])
    total_female_founders = total_female_founders[1:]
    female_stats.append(total_female_founders[1:])
    founder_number = founder_number[1:]
    # print(founder_number)


    from pywaffle import Waffle
    import matplotlib.pyplot as plt
    from matplotlib import rcParams

    if (a_sorted_number == 0):
        row_length = [5,7,6,4,1,1,1]
    else:
        row_length = [2,6,5,1,1,1,1]

    for founder_number1, female, male, row in zip(founder_number, total_female_founders, total_male_founders, row_length):


        total_male = male
        total_female = female


        rcParams["figure.dpi"] = 120
        fig = plt.figure(
            FigureClass=Waffle,
            rows= row,
            values=[female, male],
            colors=["#6D1A36", "#3F88C5"],
            icons=["female", "male"],
            font_size=12,             # was 20
            icon_style="solid",
            icon_legend=True,
            legend={
                "labels": [("female" + "(" + str(round(female/(female+male)*100)) + "%)"),
                           ("male" + "(" + str(round((male/(female+male))*100)) + "%)")],
                    # ["Female", "Male"],
                "loc": "upper left",
                "bbox_to_anchor": (1.05, 1),   # give legend a bit more breathing room
            },
        )
        if (a_sorted_number == 0):
            plt.title(str(int((female+male)/founder_number1)) +" Unicorns " + "mit " + str(founder_number1) + " Gründungsmitglieder")
        else:
            plt.title(str(int((female+male)/founder_number1)) +" (vergangenen) Unicorns " + "mit " + str(founder_number1) + " Gründungsmitglieder")
        plt.show()

In [ ]:
print(male_stats)
print(female_stats)

In [ ]:

male_stats_flat = np.array([x for sub in male_stats for x in sub]).sum()
female_stats_flat = np.array([x for sub in female_stats for x in sub]).sum()

labels = ['Männlich', 'Weiblich']
sizes = [male_stats_flat, female_stats_flat]

plt.figure(figsize=(5,5))
plt.pie(sizes, labels=labels, autopct='%1.1f%%', startangle=90, colors=["#3F88C5", "#6D1A36"],)
plt.axis('equal')
plt.title('Gründungsmitglieder nach Geschlecht')
plt.show()




In [ ]:
import matplotlib.pyplot as plt

labels = ['Männlich', 'Weiblich']
sizes = [male_stats_flat, female_stats_flat]

plt.figure(figsize=(5, 5))
bars = plt.bar(labels, sizes, color=["#3F88C5", "#6D1A36"])

plt.title('Gründungsmitglieder nach Geschlecht')
plt.ylabel('Anzahl')

# optional: show values on top of bars
for bar, val in zip(bars, sizes):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
             f'{val}', ha='center', va='bottom')

plt.tight_layout()
plt.show()


# Unfinished

In [ ]:
df = currentUnicorns_with_gender_df

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import re

# df = ... your DataFrame

# Convert to string safely
s = df["Founder_Genders"].fillna("").astype(str)

def count_genders(x):
    x = str(x).lower()
    males = len(re.findall(r'\bmale\b', x))
    females = len(re.findall(r'\bfemale\b', x))
    return pd.Series({"male_count": males, "female_count": females})

df[["male_count", "female_count"]] = s.apply(count_genders)

# Top 15 industries by total founders (male+female)
agg = df.groupby("Industry")[["male_count", "female_count"]].sum()
top15 = agg.assign(total=agg["male_count"] + agg["female_count"]).sort_values("total", ascending=False).head(15)

# Colors
colors = ["#3F88C5", "#6D1A36"]  # male, female

# Plot
ax = top15[["male_count", "female_count"]].plot(
    kind="bar",
    figsize=(12, 6),
    color=colors
)

plt.xticks(rotation=45, ha="right")
plt.xlabel("Industry")
plt.ylabel("Number of founders")
plt.title("Top 15 Industrien")
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import re

# df = ... your DataFrame

s = df["Founder_Genders"].fillna("").astype(str)

def count_genders(x):
    x = str(x).lower()
    males = len(re.findall(r'\bmale\b', x))
    females = len(re.findall(r'\bfemale\b', x))
    return pd.Series({"male_count": males, "female_count": females})

df[["male_count", "female_count"]] = s.apply(count_genders)

# Top 15 countries by total founders (male+female)
agg = df.groupby("Country/countries")[["male_count", "female_count"]].sum()
top15 = agg.assign(total=agg["male_count"] + agg["female_count"]).sort_values("total", ascending=False).head(10)

# Colors
colors = ["#3F88C5", "#6D1A36"]  # male, female

# Plot grouped bar
top15[["male_count", "female_count"]].plot(
    kind="bar",
    figsize=(12, 6),
    color=colors
)

plt.xticks(rotation=45, ha="right")
plt.xlabel("Country")
plt.ylabel("Number of founders")
plt.title("Top 10 Länder")
plt.tight_layout()
plt.show()
